# Radiotherapy: auto-segmentation model quality evaluation

## What is this notebook about?

Auto-segmentation models are increasingly used in radiotherapy planning. They can save time and reduce manual workload. But they are not infallible.

In this session, you will act as a clinical reviewer of an auto-segmentation model. You will look at model outputs, spot problems, and figure out **why** those problems occur.


By the end, you are able to anser the question what you need in order to trust a model output for a specific patient!


In [5]:
from pathlib import Path
import importlib

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False

from src.define_paths import BASE, SURGERY_EXP, METAL_EXP  # Paths to data folders
from src.roi_dictionary import ROI_DICT
from src.block2_tutorial_part2_helpers import (
    labels_to_oar_names,
    launch_metal_viewer,
    launch_postop_viewer,
    launch_pred_vs_clinical_viewer,
    load_experiment2_data,
    render_availability_table,
    render_dice_table_per_oar,
    render_dice_table_two_cases,
    render_dice_table_two_models_two_cases,
    render_oar_name_list,
 )

In [2]:
pip install nibabel

   ---------------------------------------- 0.0/3.3 MB ? eta -:--:--
   ------------------------------- -------- 2.6/3.3 MB 16.7 MB/s eta 0:00:01
   ---------------------------------------- 3.3/3.3 MB 12.2 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


---
## Before we start: how does a model learn?

Imagine teaching a new resident to contour the submandibular glands. You show them 758 example cases with correct contours. After enough examples, they get good at recognising the gland on a CT scan.

This is essentially what an auto-segmentation model does. 

**But:** the model can only learn from what it has seen.

A model is not making a mistake out of carelessness. It is doing only what it was taught, but what it was taught may not match the patient in front of you. That is what you will investigate today


---

<div style="background-color: #e8f1ff; border-left: 5px solid #2b6cb0; padding: 12px 14px; border-radius: 8px; color: #0f172a;">
<strong>Question 1:</strong> Can you think of a situation in your own clinical work where you made an assumption about an OAR or target segmentation based on what you usually see/were taught, and it turned out the patient was different from what you had seen before?
</div>

---
# Case 1
## Clinical briefing

You are reviewing the OAR auto-segmentations before starting treatment planning.

**Patient:**
- Progressive swelling right side of the neck: hypopharynx carcinoma cT1pN3M0
- Smoker (27 pack-years), alcohol intake 8 units/day
- **Treatment strategy:** radical neck dissection right side + chemoradiation of the primary tumour and bilateral neck

**Model used:**
- nnU-Net (state-of-the-art deep learning segmentation model)
- Trained on **758 primary RT patients**

## Step 1: load data

In [6]:
# Images and segmentation data
postop = nib.load(str(SURGERY_EXP["postop_image"])).get_fdata()
postop_pred = nib.load(str(SURGERY_EXP["model_pred_postop"])).get_fdata()
postop_gt = nib.load(str(SURGERY_EXP["gt_postop"])).get_fdata()

present_ids = np.unique(postop_pred)
present_names = labels_to_oar_names(present_ids, ROI_DICT)

render_oar_name_list(
    present_names,
    title="Predicted OAR names present in this case",
    n_columns=2,
)

FileNotFoundError: No such file or no access: 'C:/Users/AalstJE/OneDrive - UMCG/Documents/PhD/Code/AIOS-course/Data/processed/surgery_experiment/imagesTs/HNC-B_782_0000.nii.gz'

---

<div style="background-color: #e8f1ff; border-left: 5px solid #2b6cb0; padding: 12px 14px; border-radius: 8px; color: #0f172a;">
<strong>Question 2:</strong> In the cell above you loaded the auto-segmentation for this patient. You can see the list of structures available. Do you think that we can use one model for segmenting all these structures or need separate models?
</div>

## Step 2: inspect auto-segmentation results

In [ ]:
# Launch visualisation of image and predicted labels
launch_postop_viewer(postop=postop, postop_pred=postop_pred, roi_dict=ROI_DICT, widgets=widgets, has_widgets=HAS_WIDGETS)

Use the slider to scroll through the CT slices. You can zoom, move the view and adjust the intensity scaling The coloured overlays are the model's predicted OAR contours.

**Your task:**
- Scroll through the neck region and evaluate the segmentations for clinical correctness
- Note any contour that looks anatomically implausible

---

<div style="background-color: #e8f1ff; border-left: 5px solid #2b6cb0; padding: 12px 14px; border-radius: 8px; color: #0f172a;">
<strong>Question 3:</strong> Which structures would you send to the treatment planner without further changes?? And where should adjustments be made?
</div>

## Step 3: compare with clinically used contour
For this patient, we also have the contours available that were used clinically to make the treatment plan. 
Compare the model output against the clinical reference contour. And also see the DSC scores (overlap scores between predicted and reference contour)

Below you can see the model's prediction (what you just looked at) against the **clinical reference contour** (the version used for actual treatment planning)


The DSC score is a number between 0 and 1 that measures how well the model's contour overlaps with the clinical reference.
- **1.0** = perfect overlap
- **0.0** = no overlap at all

A score above ~0.8 is generally considered clinically acceptable for most head-and-neck OARs.

In [ ]:
# Visual comparison: prediction vs clinical contour
launch_pred_vs_clinical_viewer(postop=postop, postop_pred=postop_pred, postop_gt=postop_gt, roi_dict=ROI_DICT, widgets=widgets, has_widgets=HAS_WIDGETS)

# Per-OAR Dice scores (prediction vs clinical contour)
render_dice_table_per_oar(postop_pred=postop_pred, postop_gt=postop_gt, roi_dict=ROI_DICT)

---

<div style="background-color: #e8f1ff; border-left: 5px solid #2b6cb0; padding: 12px 14px; border-radius: 8px; color: #0f172a;">
<strong>Question 4:</strong> Did the clinical contour make changes that you did not expect?
</div>


---

<div style="background-color: #e8f1ff; border-left: 5px solid #2b6cb0; padding: 12px 14px; border-radius: 8px; color: #0f172a;">
<strong>Question 5:</strong> The DSC score for the right submandibular gland is very low. Why could that be?
</div>

- A: The model made a random error
- B: The model predicted what it would normally see, but this patient is different
- C: The Dice metric itself is misleading here

**Hint:** Take a look back at the patient and model information. Is there any discrepancy between the two that can explain the poor performance for the right submandibular?

<details>
<summary><strong>Explanation</strong></summary>

The patient had a **right radical neck dissection** — meaning the right submandibular gland was surgically removed.

The model was trained on **758 primary RT patients who had not yet had surgery**. It has never seen a post-operative neck in training. So when it processes this CT, it finds image features in the right neck region that look, to it, like they could be a submandibular gland.

The model is not broken. It is doing exactly what it was trained to do. The problem is that **this patient does not match the population the model was trained on**.

**Bigger picture: Data Completeness**

The training dataset was **complete for primary RT patients**, but it was **missing post-operative cases entirely**. That gap in the training data directly caused the error you just observed.
</details>

---
# Experiment 2
## Clinical briefing

You are again reviewing the OAR auto-segmentations before starting treatment planning.

You will compare the performance of a model on two cases and investigate why they perform differently

## Step 1: load data

In [ ]:
# Experiment 2 setup data
exp2 = load_experiment2_data(METAL_EXP, nib)

metal_img = exp2["metal_img"]
no_metal_img = exp2["no_metal_img"]
pred_model_a_case1 = exp2["pred_model_a_case1"]
pred_model_a_case2 = exp2["pred_model_a_case2"]
pred_model_b_case1 = exp2["pred_model_b_case1"]
pred_model_b_case2 = exp2["pred_model_b_case2"]
gt_case1 = exp2["gt_case1"]
gt_case2 = exp2["gt_case2"]

## Step 2: inspect auto-segmentation results

### Case 1

In [ ]:
launch_postop_viewer(postop=metal_img, postop_pred=pred_model_a_case1, roi_dict=ROI_DICT, widgets=widgets, has_widgets=HAS_WIDGETS)

Use the slider to scroll through the CT slices. You can zoom, move the view and adjust the intensity scaling The coloured overlays are the model's predicted OAR contours.

**Your task:**
- Scroll through the neck region and evaluate the segmentations for clinical correctness
- Note any contour that looks anatomically implausible

---

<div style="background-color: #e8f1ff; border-left: 5px solid #2b6cb0; padding: 12px 14px; border-radius: 8px; color: #0f172a;">
<strong>Question 6:</strong> Which structures would you send to the treatment planner without further changes?? And where should adjustments be made?
</div>

### Case 2
Now review **Case 2** with the same model and compare it to your findings from Case 1.

In [ ]:
launch_postop_viewer(postop=no_metal_img, postop_pred=pred_model_a_case2, roi_dict=ROI_DICT, widgets=widgets, has_widgets=HAS_WIDGETS)

---

<div style="background-color: #e8f1ff; border-left: 5px solid #2b6cb0; padding: 12px 14px; border-radius: 8px; color: #0f172a;">
<strong>Question 7:</strong> Which structures would you send to the treatment planner without further changes?? And where should adjustments be made? Are there structures where the model performs better on for this case?
</div>

## Step 3: compare with clinically used contour
For these two patients, we again also have the contours available that were used clinically to make the treatment plan. 
Compare the model output against the clinical reference contour. And also see the DSC scores (overlap scores between predicted and reference contour)

Below you can see the model's predictions (what you just looked at) against the **clinical reference contour** (the version used for actual treatment planning)


The DSC score is a number between 0 and 1 that measures how well the model's contour overlaps with the clinical reference.
- **1.0** = perfect overlap
- **0.0** = no overlap at all

A score above ~0.8 is generally considered clinically acceptable for most head-and-neck OARs.

In [ ]:
print("Case 1")
launch_pred_vs_clinical_viewer(postop=metal_img, postop_pred=pred_model_a_case1, postop_gt=gt_case1, roi_dict=ROI_DICT, widgets=widgets, has_widgets=HAS_WIDGETS)

print("Case 2")

launch_pred_vs_clinical_viewer(postop=no_metal_img, postop_pred=pred_model_a_case2, postop_gt=gt_case2, roi_dict=ROI_DICT, widgets=widgets, has_widgets=HAS_WIDGETS)


render_dice_table_two_cases(pred_case1=pred_model_a_case1, pred_case2=pred_model_a_case2, gt_case1=gt_case1, gt_case2=gt_case2, roi_dict=ROI_DICT, case1_name="Case 1",case2_name="Case 2")



---

<div style="background-color: #e8f1ff; border-left: 5px solid #2b6cb0; padding: 12px 14px; border-radius: 8px; color: #0f172a;">
<strong>Question 8:</strong> What is the problem with case 1 compared to case 2? What is special about the visual appearance of case 1 that disrupts model performance?
</div>

Hint 1: small structures (such as the arytenoids) have a misleading DSC score because that score is dependent on volume. \
Hint 2: The mandible is normally one of the easiest structures to segment due to its distinct bone intensity. Performance is usually >0.90, why is that not the case in Model A for case 1

<details>
<summary><strong>Explanation</strong></summary>

Case 1 contains **dental metal artefacts** (streaking) that distort the CT appearance of nearby anatomy.

Model A was trained only on cases **without metal artefacts**. So when it sees Case 1, key visual cues it normally uses (intensity patterns and structure boundaries) are disrupted.

The model is not broken. It is doing what it was trained to do, but the image characteristics in this patient do not match its training experience.

Model B performs better here because it was trained on both artefact and non-artefact scans, so it learned to handle this type of distortion.

**Bigger picture: Data Fidelity**

Good model performance depends on training data that reflects real-world image quality and acquisition conditions. If artefact-heavy scans are missing from training, performance can drop sharply when those scans appear in clinical practice.
</details>